# Chapter 13. Kinematic candidates and multimodal gates

**Learning objective.** Preserve all candidate landmarks, distinguish phonetic nonidentifiability from software failure, quantify profile-dependent eligibility, and issue stream-specific and combined multimodal decisions.

**Estimated time:** 90–120 minutes. **Exercises:** 13.1 and 13.4. The smooth, plateaued, multi-peaked, noisy, missing, and low-excursion trajectories are generated teaching cases, not observations of speech movement.

## Input and output contract

Inputs are the synthetic trajectories, four prespecified filtering-and-threshold profiles, and the preceding stream-quality decisions. Outputs retain every velocity candidate, one rule-governed landmark record per event and profile, availability by trajectory class, profile sensitivity, diagnostics, a multimodal gate, and provenance.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    if (candidate / "companion" / "data" / "MANIFEST.tsv").is_file():
        REPOSITORY_ROOT = candidate
        break
else:
    raise RuntimeError("Run from the repository root or one of its subdirectories.")
SOURCE_ROOT = REPOSITORY_ROOT / "companion" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
DATA_ROOT = REPOSITORY_ROOT / "companion" / "data" / "ch13" / "synthetic_multimodal"
if not DATA_ROOT.exists():
    raise FileNotFoundError("Chapter 13 fixture absent. Run companion/scripts/generate_ch13_fixture.py explicitly; no data are substituted silently.")
print("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — generated geometry, clocks, and trajectories, not human articulatory data.")

In [ ]:
from phonetic_research_companion.ch13 import run_kinematic_landmarks
QUALITY = REPOSITORY_ROOT / "companion" / "outputs" / "ch13" / "01_coordinate_and_sync_audit" / "stream_quality.tsv"
if not QUALITY.exists():
    raise FileNotFoundError("Run 01_coordinate_and_sync_audit.ipynb first; stream decisions are not silently replaced.")
OUTPUT = REPOSITORY_ROOT / "companion" / "outputs" / "ch13" / "02_kinematic_landmarks"
result = run_kinematic_landmarks(DATA_ROOT / "validated_trajectories.tsv", DATA_ROOT / "landmark_profiles.yaml", QUALITY, OUTPUT)
assert result["raw_unchanged"]
assert result["candidates"]
assert any(row["landmark_status"] == "phonetic_nonidentifiability" for row in result["landmarks"])
assert any(row["landmark_status"].startswith("eligible") for row in result["landmarks"])
for row in result["availability"]:
    print(row)

## Learner decision

Build an observability matrix before interpreting a multimodal contrast. Then compare the four landmark profiles for smooth, plateaued, multi-peaked, noisy, missing, and low-excursion trajectories. Explain why a numeric landmark is not forced for every event, how the eligible population changes, and why one global “reliability” label would conceal incompatible stream-level decisions.

In [ ]:
required = ["kinematic_candidates.tsv", "kinematic_landmarks.tsv", "landmark_availability.tsv", "landmark_sensitivity.tsv", "kinematic_diagnostics.html", "multimodal_gate.yaml", "run_provenance.json"]
assert all((OUTPUT / name).exists() for name in required)
assert result["gate"]["combined_decision"] == "restrict"
assert not result["gate"]["one_global_reliability_label_permitted"]
print(result["gate"])